In [1]:
# Instalasi otomatis Sastrawi jika belum tersedia.
# Di Google Colab/Jupyter yang memiliki akses internet, sel ini akan memasang library secara otomatis.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("Sastrawi") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "Sastrawi", "-q"])

import re
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

pd.set_option("display.max_colwidth", None)

In [2]:
# Dataset minimal 5 dokumen Bahasa Indonesia
documents = [
    "Fakultas Teknik UNM mengembangkan sistem pencarian dokumen berbasis teknologi informasi.",
    "Mahasiswa Teknik Komputer mempelajari jaringan komputer dan keamanan sistem secara bertahap.",
    "Kecerdasan buatan membantu komputer mengenali pola data dan memberikan prediksi yang berguna.",
    "Sistem temu kembali informasi digunakan untuk menemukan dokumen yang sesuai dengan kebutuhan pengguna.",
    "Perkembangan teknologi digital mendorong pendidikan menggunakan platform pembelajaran yang lebih interaktif."
]

df = pd.DataFrame({
    "Dokumen": [f"Dokumen {i}" for i in range(1, 6)],
    "Teks Asli": documents
})

df

,Dokumen,Teks Asli
0,Dokumen 1,Fakultas Teknik UNM mengembangkan sistem pencarian dokumen berbasis teknologi informasi.
1,Dokumen 2,Mahasiswa Teknik Komputer mempelajari jaringan komputer dan keamanan sistem secara bertahap.
2,Dokumen 3,Kecerdasan buatan membantu komputer mengenali pola data dan memberikan prediksi yang berguna.
3,Dokumen 4,Sistem temu kembali informasi digunakan untuk menemukan dokumen yang sesuai dengan kebutuhan pengguna.
4,Dokumen 5,Perkembangan teknologi digital mendorong pendidikan menggunakan platform pembelajaran yang lebih interaktif.


In [3]:
# Inisialisasi Stopword Remover dan Stemmer Sastrawi
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def preprocess_text(text):
    # 1. Case folding
    text = text.lower()

    # 2. Cleaning: hapus angka, tanda baca, dan karakter khusus
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # 3. Tokenisasi
    tokens = text.split()

    # 4. Stopwords removal menggunakan Sastrawi
    filtered_text = stopword_remover.remove(" ".join(tokens))
    tokens = filtered_text.split()

    # 5. Stemming menggunakan Sastrawi
    stemmed_text = stemmer.stem(" ".join(tokens))
    stemmed_tokens = stemmed_text.split()

    return stemmed_tokens

# Fungsi untuk melihat token setelah tiap tahap
def preprocessing_detail(text):
    casefolded = text.lower()
    cleaned = re.sub(r"[^a-zA-Z\s]", " ", casefolded)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    tokens = cleaned.split()
    without_stopwords = stopword_remover.remove(" ".join(tokens))
    stopword_tokens = without_stopwords.split()
    stemmed = stemmer.stem(" ".join(stopword_tokens))
    stemmed_tokens = stemmed.split()

    return {
        "case_folding": casefolded,
        "cleaning": cleaned,
        "tokenisasi": tokens,
        "stopwords_removal": stopword_tokens,
        "stemming": stemmed_tokens
    }

In [4]:
# Terapkan preprocessing ke seluruh dokumen
df["Token Sebelum"] = df["Teks Asli"].apply(lambda x: x.split())
df["Token Setelah"] = df["Teks Asli"].apply(preprocess_text)
df["Teks Setelah"] = df["Token Setelah"].apply(lambda x: " ".join(x))

df[["Dokumen", "Teks Asli", "Teks Setelah"]]

,Dokumen,Teks Asli,Teks Setelah
0,Dokumen 1,Fakultas Teknik UNM mengembangkan sistem pencarian dokumen berbasis teknologi informasi.,fakultas teknik unm kembang sistem cari dokumen bas teknologi informasi
1,Dokumen 2,Mahasiswa Teknik Komputer mempelajari jaringan komputer dan keamanan sistem secara bertahap.,mahasiswa teknik komputer ajar jaring komputer aman sistem tahap
2,Dokumen 3,Kecerdasan buatan membantu komputer mengenali pola data dan memberikan prediksi yang berguna.,cerdas buat bantu komputer nali pola data beri prediksi guna
3,Dokumen 4,Sistem temu kembali informasi digunakan untuk menemukan dokumen yang sesuai dengan kebutuhan pengguna.,sistem temu informasi guna temu dokumen sesuai butuh guna
4,Dokumen 5,Perkembangan teknologi digital mendorong pendidikan menggunakan platform pembelajaran yang lebih interaktif.,kembang teknologi digital dorong didik guna platform ajar lebih interaktif


## Perbandingan sebelum dan sesudah preprocessing

Dua dokumen pertama ditampilkan lebih lengkap agar setiap tahapan dapat diamati.

In [5]:
for i in range(2):
    print("=" * 80)
    print(df.loc[i, "Dokumen"])
    print("Teks asli        :", df.loc[i, "Teks Asli"])
    detail = preprocessing_detail(df.loc[i, "Teks Asli"])
    print("Case folding     :", detail["case_folding"])
    print("Cleaning         :", detail["cleaning"])
    print("Tokenisasi       :", detail["tokenisasi"])
    print("Stopwords removal:", detail["stopwords_removal"])
    print("Stemming         :", detail["stemming"])

Dokumen 1
Teks asli        : Fakultas Teknik UNM mengembangkan sistem pencarian dokumen berbasis teknologi informasi.
Case folding     : fakultas teknik unm mengembangkan sistem pencarian dokumen berbasis teknologi informasi.
Cleaning         : fakultas teknik unm mengembangkan sistem pencarian dokumen berbasis teknologi informasi
Tokenisasi       : ['fakultas', 'teknik', 'unm', 'mengembangkan', 'sistem', 'pencarian', 'dokumen', 'berbasis', 'teknologi', 'informasi']
Stopwords removal: ['fakultas', 'teknik', 'unm', 'mengembangkan', 'sistem', 'pencarian', 'dokumen', 'berbasis', 'teknologi', 'informasi']
Stemming         : ['fakultas', 'teknik', 'unm', 'kembang', 'sistem', 'cari', 'dokumen', 'bas', 'teknologi', 'informasi']
Dokumen 2
Teks asli        : Mahasiswa Teknik Komputer mempelajari jaringan komputer dan keamanan sistem secara bertahap.
Case folding     : mahasiswa teknik komputer mempelajari jaringan komputer dan keamanan sistem secara bertahap.
Cleaning         : mahasiswa teknik

In [6]:
# Ringkasan token sebelum, sesudah, dan persentase pengurangan
summary = pd.DataFrame({
    "Dokumen": df["Dokumen"],
    "Jumlah Token Sebelum": df["Token Sebelum"].apply(len),
    "Jumlah Token Setelah": df["Token Setelah"].apply(len)
})

summary["Pengurangan Token"] = (
    summary["Jumlah Token Sebelum"] - summary["Jumlah Token Setelah"]
)
summary["Persentase Pengurangan (%)"] = (
    summary["Pengurangan Token"] / summary["Jumlah Token Sebelum"] * 100
).round(2)

summary

,Dokumen,Jumlah Token Sebelum,Jumlah Token Setelah,Pengurangan Token,Persentase Pengurangan (%)
0,Dokumen 1,10,10,0,0.00
1,Dokumen 2,11,9,2,18.18
2,Dokumen 3,12,10,2,16.67
3,Dokumen 4,13,9,4,30.77
4,Dokumen 5,11,10,1,9.09


Hasil Analisis

Preprocessing membuat data teks menjadi lebih bersih dan konsisten sehingga lebih siap digunakan dalam sistem temu kembali informasi. Case folding menyamakan bentuk huruf, cleaning menghilangkan angka serta tanda baca yang tidak diperlukan, tokenisasi memecah teks menjadi unit kata, stopwords removal mengurangi kata umum yang kurang membedakan dokumen, sedangkan stemming menyamakan kata berimbuhan ke bentuk dasarnya. Dampaknya, jumlah token menjadi lebih sedikit dan representasi kata lebih terfokus pada istilah yang membawa informasi, sehingga proses pencarian dan perhitungan kemiripan dokumen dapat menjadi lebih efisien. Namun, penghapusan stopwords dan stemming tetap perlu diperhatikan karena pada konteks tertentu dapat menghilangkan informasi yang masih relevan.